inicializamos 3 dataframes

In [2]:
import sys
from pathlib import Path

# El notebook puede estar en trabajo/; subimos hasta encontrar el repo.
_here = Path.cwd().resolve()
ROOT = next(
    p
    for p in [_here, *_here.parents]
    if (p / "labs" / "_shared" / "session.py").is_file()
)
sys.path.insert(0, str(ROOT / "labs" / "_shared"))

from paths import RAW, STAGING, CURATED  # rutas absolutas, no Path("data/raw")
from session import get_spark

print("ROOT   ", ROOT)
print("RAW    ", RAW, "existe:", RAW.is_dir())
print("STAGING", STAGING)
print("CURATED", CURATED)


from pyspark.sql.functions import col, date_format

spark = get_spark("novashop-m03")
orders = spark.read.parquet(str(STAGING / "orders_clean"))
items = spark.read.parquet(str(STAGING / "order_items_clean"))
lines = (
    items.join(orders, "order_id", "inner")
    .withColumn("gmv_line", col("qty") * col("unit_price") * (1 - col("discount")))
    .withColumn("order_month", date_format(col("order_ts"), "yyyy-MM"))
)
print(lines.count(), lines.where(col("gmv_line") < 0).count())


ROOT    /workspaces/python-pyspark-201
RAW     /workspaces/python-pyspark-201/data/raw existe: True
STAGING /workspaces/python-pyspark-201/data/staging
CURATED /workspaces/python-pyspark-201/data/curated
1980 13


In [ ]:
from pyspark.sql.functions import when, lower, least, lit

fact = (
    lines.withColumn("discount", least(col("discount"), lit(1.0))) .withColumn(
        "channel_norm",
        when(lower(col("channel")).isin("web", "app", "store"), lower(col("channel")))
        .otherwise(lit("other")),
    ) .withColumn("is_billable", col("status") == "paid") .withColumn(
        "gmv_line",
        col("qty") * col("unit_price") * (1 - col("discount")),
    )
)

print("filas", fact.count())
print("gmv < 0", fact.where(col("gmv_line") < 0).count())


filas 1980
gmv < 0 0


Validamos el dominio

In [14]:
fact.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (9)
+- Project (8)
   +- Project (7)
      +- BroadcastHashJoin Inner BuildRight (6)
         :- Filter (2)
         :  +- Scan parquet  (1)
         +- BroadcastExchange (5)
            +- Filter (4)
               +- Scan parquet  (3)


(1) Scan parquet 
Output [5]: [order_id#96, product_id#97, qty#98, unit_price#99, discount#100]
Batched: true
Location: InMemoryFileIndex [file:/workspaces/python-pyspark-201/data/staging/order_items_clean]
PushedFilters: [IsNotNull(order_id)]
ReadSchema: struct<order_id:string,product_id:string,qty:int,unit_price:decimal(10,2),discount:decimal(5,2)>

(2) Filter
Input [5]: [order_id#96, product_id#97, qty#98, unit_price#99, discount#100]
Condition : isnotnull(order_id#96)

(3) Scan parquet 
Output [5]: [order_id#86, customer_id#87, status#88, channel#89, order_ts#90]
Batched: true
Location: InMemoryFileIndex [file:/workspaces/python-pyspark-201/data/staging/orders_clean]
PushedFilters: [IsNotNull(order_id)]
ReadSc

In [11]:
fact.groupBy("channel_norm").count().orderBy("channel_norm").show()
print("discount > 1", fact.where(col("discount") > 1).count())
print("billable", fact.where(col("is_billable")).count())

+------------+-----+
|channel_norm|count|
+------------+-----+
|         app|  715|
|       other|  247|
|       store|  243|
|         web|  775|
+------------+-----+

discount > 1 0
billable 1127


Escribimos un fichero en Stag con esto

In [12]:
dest = STAGING / "fact_lines"
fact.write.mode("overwrite").parquet(str(dest))
print(spark.read.parquet(str(dest)).count())


1980


In [13]:
fact.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (9)
+- Project (8)
   +- Project (7)
      +- BroadcastHashJoin Inner BuildRight (6)
         :- Filter (2)
         :  +- Scan parquet  (1)
         +- BroadcastExchange (5)
            +- Filter (4)
               +- Scan parquet  (3)


(1) Scan parquet 
Output [5]: [order_id#96, product_id#97, qty#98, unit_price#99, discount#100]
Batched: true
Location: InMemoryFileIndex [file:/workspaces/python-pyspark-201/data/staging/order_items_clean]
PushedFilters: [IsNotNull(order_id)]
ReadSchema: struct<order_id:string,product_id:string,qty:int,unit_price:decimal(10,2),discount:decimal(5,2)>

(2) Filter
Input [5]: [order_id#96, product_id#97, qty#98, unit_price#99, discount#100]
Condition : isnotnull(order_id#96)

(3) Scan parquet 
Output [5]: [order_id#86, customer_id#87, status#88, channel#89, order_ts#90]
Batched: true
Location: InMemoryFileIndex [file:/workspaces/python-pyspark-201/data/staging/orders_clean]
PushedFilters: [IsNotNull(order_id)]
ReadSc